<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

In [1]:
from IPython.display import Markdown, display
import json
from collections import Counter
from datetime import datetime
from zoneinfo import ZoneInfo
import pytz
print('ॐ श्री सरस्वत्यै नमः',datetime.now(pytz.timezone('Asia/Calcutta')))

ॐ श्री सरस्वत्यै नमः 2026-10-12 13:48:15.995155+05:30


In [2]:
!wget -O CentaurLogo.png -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/logo-S.png
!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py
!wget -O mosClassifier.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosClassifier.py


from mos import *
from mosClassifier import *


In [4]:
client = authenticateOpenAI()
model = "gpt-5.6-luna"

# Real-world evidence, LLM roles and prompts
# are stored persistently in Google Drive.

from google.colab import drive
drive.mount('/content/drive')

MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
#OP_PATH = f"{MOS_HOME}/Reports"
ARCHIVE_PATH = f"{MOS_HOME}/Archived_Scenarios"
CL_PATH = f"{MOS_HOME}/Classifier"

#!ls -l "{MOS_HOME}"
#!ls -lh "{OP_PATH}"
!ls -lh "{ARCHIVE_PATH}"
!ls -lh "{CL_PATH}"

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 16K
drwx------ 2 root root 4.0K Oct 12 07:50 TR1ME3_20
drwx------ 2 root root 4.0K Oct 12 07:59 TR1ME3_40
drwx------ 2 root root 4.0K Oct 12 07:50 TR2ME3_20
drwx------ 2 root root 4.0K Oct 12 07:59 TR2ME3_40
total 5.0K
-rw------- 1 root root 3.0K Oct 11 10:31 CL_3Way_v0.txt
-rw------- 1 root root 1.9K Oct 11 10:31 CL_Generic_Role_v0.txt


In [6]:
# ---------------------------------------------------------
# Load Classifier Role Files
# ---------------------------------------------------------

CL_Generic_Role = readRoleFile(f"{CL_PATH}/CL_Generic_Role_v0.txt")
CL_3Way = readRoleFile(f"{CL_PATH}/CL_3Way_v0.txt")

# ---------------------------------------------------------
# Initialise Classifier
# ---------------------------------------------------------

Classifier = Classifier_Agent(
    client,
    model,
    "Classifier_3Way",
    CL_Generic_Role
)

In [8]:
def Classify(TargetFolder):

    # ---------------------------------------------------------
    # Load Structured Scenario Reports
    # ---------------------------------------------------------
    Reports = readMarkdownFiles(f"{ARCHIVE_PATH}/{TargetFolder}")
    print(f"Loaded {len(Reports)} scenario reports from {TargetFolder}")

    #{
    #    "MOS_TR2ME3_R01_....md": "Complete contents of report 1",
    #    "MOS_TR2ME3_R02_....md": "Complete contents of report 2",
    #    ...
    #}

    # ---------------------------------------------------------
    # Classify Each Scenario
    # ---------------------------------------------------------

    Classifications = {}

    for filename, report in Reports.items():

        print(f"\nClassifying: {filename}")

        Classifications[filename] = Classifier.ClassifyScenario(
            report,
            CL_3Way
        )


    #{
    #    "MOS_TR2ME3_R01_....md": '{"classification":"De-escalation","rationale":"..."}',
    #    "MOS_TR2ME3_R02_....md": '{"classification":"Escalation","rationale":"..."}',
    #    ...
    #}

    return Classifications




In [25]:
#Classification_TR1ME3_40 = Classify("TR1ME3_40")

In [10]:
import json
from pathlib import Path

OutputFile = Path(ARCHIVE_PATH) / "Classification_TR1ME3_40.json"

with open(OutputFile, "w", encoding="utf-8") as f:
    json.dump(Classification_TR1ME3_40, f, indent=2, ensure_ascii=False)

print(f"Saved {len(Classification_TR1ME3_40)} classifications to {OutputFile}")

Saved 40 classifications to /content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/Archived_Scenarios/Classification_TR1ME3_40.json


In [14]:
def ShowClassifications(Classifications):

    Counts = Counter()

    for filename, result in Classifications.items():
        result = json.loads(result)

        category = result["classification"]
        Counts[category] += 1

        #print(f"\n{filename}, {category}\n{result['rationale']}")
        print(f"\n{filename}, {category}")

    # ---------------------------------------------------------
    # Classification Summary
    # ---------------------------------------------------------

    total = sum(Counts.values())

    print("\nCLASSIFICATION SUMMARY")
    print("-" * 45)

    for category, count in Counts.most_common():
        print(f"{category:20s} : {count:3d} ({count/total:5.1%})")

    print("-" * 45)
    print(f"{'TOTAL':20s} : {total:3d}")

In [30]:
#ShowClassifications(Classification_TR1ME3_40)

In [26]:
#Classification_TR2ME3_40 = Classify("TR2ME3_40")

In [17]:
import json
from pathlib import Path

OutputFile = Path(ARCHIVE_PATH) / "Classification_TR2ME3_40.json"

with open(OutputFile, "w", encoding="utf-8") as f:
    json.dump(Classification_TR2ME3_40, f, indent=2, ensure_ascii=False)

print(f"Saved {len(Classification_TR2ME3_40)} classifications to {OutputFile}")

Saved 40 classifications to /content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/Archived_Scenarios/Classification_TR2ME3_40.json


In [29]:
#ShowClassifications(Classification_TR2ME3_40)

In [21]:
def getArchivedClassification(filename):

    archiveFile = Path(ARCHIVE_PATH) / filename
    with open(archiveFile, "r", encoding="utf-8") as f:
        archivedClassification = json.load(f)

    print(f"Loaded {len(archivedClassification)} classifications")
    return archivedClassification

In [27]:
ShowClassifications(getArchivedClassification("Classification_TR1ME3_40.json"))

Loaded 40 classifications

MOS_TR1ME3_R01_261009_165710.md, Escalation

MOS_TR1ME3_R02_261009_165827.md, Escalation

MOS_TR1ME3_R03_261009_165940.md, Escalation

MOS_TR1ME3_R04_261009_170058.md, Escalation

MOS_TR1ME3_R05_261009_170213.md, Escalation

MOS_TR1ME3_R06_261009_170332.md, Escalation

MOS_TR1ME3_R07_261009_170444.md, Escalation

MOS_TR1ME3_R08_261009_170556.md, De-escalation

MOS_TR1ME3_R09_261009_170715.md, Escalation

MOS_TR1ME3_R10_261009_170831.md, Escalation

MOS_TR1ME3_R11_261009_170947.md, Escalation

MOS_TR1ME3_R12_261009_171100.md, Escalation

MOS_TR1ME3_R13_261009_171213.md, Escalation

MOS_TR1ME3_R14_261009_171326.md, Escalation

MOS_TR1ME3_R15_261009_171443.md, Escalation

MOS_TR1ME3_R16_261009_171603.md, Escalation

MOS_TR1ME3_R17_261009_171722.md, Escalation

MOS_TR1ME3_R18_261009_171840.md, Stalemate

MOS_TR1ME3_R19_261009_172000.md, Escalation

MOS_TR1ME3_R20_261009_172119.md, Escalation

MOS_TR1ME3_R21_261012_115653.md, Escalation

MOS_TR1ME3_R22_261012_1158

In [28]:
ShowClassifications(getArchivedClassification("Classification_TR2ME3_40.json"))

Loaded 40 classifications

MOS_TR2ME3_R01_261009_190032.md, De-escalation

MOS_TR2ME3_R02_261009_190126.md, Stalemate

MOS_TR2ME3_R03_261009_190223.md, Stalemate

MOS_TR2ME3_R04_261009_190314.md, De-escalation

MOS_TR2ME3_R05_261009_190411.md, De-escalation

MOS_TR2ME3_R06_261009_190506.md, De-escalation

MOS_TR2ME3_R07_261009_190605.md, De-escalation

MOS_TR2ME3_R08_261009_190704.md, De-escalation

MOS_TR2ME3_R09_261009_190801.md, Stalemate

MOS_TR2ME3_R10_261009_190900.md, De-escalation

MOS_TR2ME3_R11_261009_190956.md, De-escalation

MOS_TR2ME3_R12_261009_191053.md, Stalemate

MOS_TR2ME3_R13_261009_191153.md, De-escalation

MOS_TR2ME3_R14_261009_191256.md, De-escalation

MOS_TR2ME3_R15_261009_191358.md, De-escalation

MOS_TR2ME3_R16_261009_191503.md, Stalemate

MOS_TR2ME3_R17_261009_191610.md, Stalemate

MOS_TR2ME3_R18_261009_191718.md, De-escalation

MOS_TR2ME3_R19_261009_191822.md, Stalemate

MOS_TR2ME3_R20_261009_191932.md, De-escalation

MOS_TR2ME3_R21_261012_123014.md, De-escal